Libraries and Setup

In [19]:
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, GRU, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)
print('Libraries imported for Optimized DL Training')

TensorFlow version: 2.20.0
Libraries imported for Optimized DL Training


Data Loading & Reshaping

In [20]:
X_train_scaled = np.load('models/X_train_scaled.npy')
y_train_scaled = np.load('models/y_train_scaled.npy')
X_test_scaled = np.load('models/X_test_scaled.npy')
y_test_scaled = np.load('models/y_test_scaled.npy')

scaler_y = joblib.load('models/scaler_y.pkl')

X_train_dl = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
X_test_dl = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])

print(f"Data Reshaped for Global Panel Model:")
print(f"Training Shape : {X_train_dl.shape}") 
print(f"Testing Shape  : {X_test_dl.shape}")

Data Reshaped for Global Panel Model:
Training Shape : (1464, 1, 12)
Testing Shape  : (366, 1, 12)


Bi-LSTM Model

In [21]:
def build_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(64)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1) 
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("Training Bi-LSTM")
lstm_model = build_lstm_model((1, X_train_scaled.shape[1])) 

history_lstm = lstm_model.fit(
    X_train_dl, y_train_scaled, 
    epochs=150, batch_size=32, 
    validation_data=(X_test_dl, y_test_scaled),
    callbacks=[early_stop],
    verbose=0
)
print("Bi-LSTM Training Complete")

Training Bi-LSTM
Bi-LSTM Training Complete


GRU Model

In [22]:
def build_stable_gru(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        GRU(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid') 
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

print("Training Stable GRU")

gru_model = build_stable_gru((1, X_train_scaled.shape[1]))

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_gru = gru_model.fit(
    X_train_dl, y_train_scaled, 
    epochs=100, batch_size=32, 
    validation_data=(X_test_dl, y_test_scaled),
    callbacks=[early_stop],
    verbose=0
)
print("GRU Training Complete")

Training Stable GRU
GRU Training Complete


Advanced DL Evaluation

In [23]:
score_lstm = evaluate_real_world_dl("Bi-LSTM", lstm_model, X_test_dl, y_test_scaled)

score_gru = evaluate_real_world_dl("GRU", gru_model, X_test_dl, y_test_scaled)

Bi-LSTM Performance 
   R2 Score : 0.9702
   MAPE     : 9.74%
   RMSE     : 730.41

GRU Performance 
   R2 Score : 0.8417
   MAPE     : 24.34%
   RMSE     : 1682.72



Saving DL Models

In [24]:
if not os.path.exists('models'):
    os.makedirs('models')

lstm_model.save('models/BiLSTM_model.h5')
gru_model.save('models/GRU_model.h5')

print("Super Stable Models saved successfully:")
print(" -> models/BiLSTM_model.h5")
print(" -> models/GRU_model.h5")

Super Stable Models saved successfully:
 -> models/BiLSTM_model.h5
 -> models/GRU_model.h5
